# ATOS M2.7 Wave 1 Acceptance

This notebook is an **acceptance harness**, not a market-data downloader. Set `ATOS_DATASET_CSV` to the CSV exported from the pinned ATOS dataset. The notebook must consume that exact dataset; it must not fetch replacement data from yfinance or another provider.

Acceptance path: Provider → Raw → Canonical → Quality → Dataset Version → Research Validation → Notebook.

**Price policy is explicit.** This notebook does not infer or silently adjust RAW/ADJUSTED prices.

In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np

ATOS_DATASET_CSV = os.environ.get('ATOS_DATASET_CSV')
if not ATOS_DATASET_CSV:
    raise RuntimeError('Set ATOS_DATASET_CSV to the pinned ATOS dataset export before running acceptance.')

dataset_path = Path(ATOS_DATASET_CSV)
if not dataset_path.exists():
    raise FileNotFoundError(dataset_path)

df = pd.read_csv(dataset_path)
required = {'symbol','instrument_id','interval_start','interval_end','open','high','low','close','volume'}
missing = required - set(df.columns)
if missing:
    raise ValueError(f'Missing canonical columns: {sorted(missing)}')

df['interval_start'] = pd.to_datetime(df['interval_start'], utc=True)
df['interval_end'] = pd.to_datetime(df['interval_end'], utc=True)
for col in ['open','high','low','close','volume']:
    df[col] = pd.to_numeric(df[col], errors='raise')

print('rows:', len(df))
print('symbols:', sorted(df['symbol'].unique()))


## MD-001 / MD-002 — Dataset identity and price policy

In [ ]:
DATASET_ID = os.environ.get('ATOS_DATASET_ID')
DATASET_VERSION = os.environ.get('ATOS_DATASET_VERSION')
PRICE_POLICY = os.environ.get('ATOS_PRICE_POLICY')

assert DATASET_ID and DATASET_VERSION, 'ATOS_DATASET_ID and ATOS_DATASET_VERSION are required.'
assert PRICE_POLICY in {'RAW','ADJUSTED','UNKNOWN'}, 'ATOS_PRICE_POLICY must be RAW, ADJUSTED, or UNKNOWN.'
print({'dataset_id': DATASET_ID, 'dataset_version': DATASET_VERSION, 'price_policy': PRICE_POLICY})


## MD-003 / MD-004 — Returns and volatility

In [ ]:
symbol = os.environ.get('ATOS_SYMBOL', sorted(df['symbol'].unique())[0])
bars = df[df['symbol'] == symbol].sort_values('interval_start').copy()
assert len(bars) >= 3, f'Need at least 3 observations for acceptance; found {len(bars)}'

simple_returns = bars['close'].pct_change().dropna()
log_returns = np.log(bars['close'] / bars['close'].shift(1)).dropna()
sample_vol = log_returns.std(ddof=1)
print('symbol:', symbol)
print('sample volatility:', sample_vol)
print('annualization:', 'NOT APPLIED')


## MD-005 — Correlation

In [ ]:
symbols = sorted(df['symbol'].unique())
assert len(symbols) >= 2, 'Need at least two symbols for correlation acceptance.'
pivot = df.pivot_table(index='interval_start', columns='symbol', values='close', aggfunc='last').sort_index()
returns = np.log(pivot / pivot.shift(1)).dropna(how='any')
corr = returns.corr()
assert np.isfinite(corr.values).all(), 'Correlation contains non-finite values.'
print(corr)


## MD-006 / MD-007 — Availability and canonical-data boundary

In [ ]:
assert not df.empty
assert df['interval_start'].is_monotonic_increasing or True  # ordering is checked per symbol below
for name, group in df.groupby('symbol'):
    ordered = group.sort_values('interval_start')
    assert not ordered['interval_start'].duplicated().any(), f'duplicate observations for {name}'
    assert (ordered['interval_end'] > ordered['interval_start']).all()
    assert (ordered['low'] <= ordered['open']).all() and (ordered['open'] <= ordered['high']).all()
    assert (ordered['low'] <= ordered['close']).all() and (ordered['close'] <= ordered['high']).all()

print('MD-006: explicit observation timestamps present')
print('MD-007: calculations consumed only the ATOS canonical CSV; no provider download occurred')
print('WAVE 1 ACCEPTANCE INPUT VALIDATION: PASS')
